# Orb：一种快速、可扩展的神经网络势能模型

## 模型简介

Orb 是一种通用的机器学习原子间势（MLFF），采用可扩展的图神经网络架构，通过并行化设计实现高效的材料建模。该模型无需依赖旋转等变性约束，仅通过数据驱动学习原子间复杂相互作用，即可在几何优化、蒙特卡洛和分子动力学模拟等多种任务中达到从头算精度。在 Matbench Discovery 基准上发布时误差降低 31%，并在大系统规模下比现有开源模型快 3-6 倍，支持零样本高温度非周期分子模拟的长期稳定运行。

## 模型架构

Orb 的核心为增强型图网络模拟器（GNS），结合平滑图注意力机制与距离截断函数构建消息传递。

* 图构建：原子系统表示为 G=(V,E,C)，节点嵌入仅含原子类型，边特征融合归一化位移向量与高斯基 RBF 展开，支持周期边界条件；
* 三阶段处理：编码器初始化节点与边特征；处理器堆叠消息传递层，通过残差边更新与双向消息聚合实现特征交互，并引入 sigmoid 加权截断门确保连续性；
* 解码器：独立 MLP 头并行预测总能量、逐原子力与晶胞应力，推理阶段通过净零力和净零扭矩后处理保证物理一致性。

## 应用场景

Orb 适用于多种高通量原子尺度模拟任务，包括：

* 晶体结构的高精度几何优化与稳定性预测，如 Matbench Discovery 基准中的分解能评估；
* 大规模分子动力学模拟，支持上千原子系统长时间稳定运行，用于扩散、掺杂等稀疏统计现象研究；
* 复杂多孔材料吸附行为建模，如 MOF 中 CO₂ 低压吸附自由能面与吸附热计算，结合 D3 校正实现范德瓦尔斯相互作用精确描述。

> 参考论文："Orb: A Fast, Scalable Neural Network Potential"

In [ ]:
!pip install -r requirement.txt

In [ ]:
import logging
import warnings
import os
import timeit
from typing import Dict, Optional

import mindspore as ms
from mindspore import nn, ops, context
import mindspore.dataset as ds
from mindspore.communication import init
from mindspore.communication import get_rank, get_group_size

from src import base, pretrained, utils
from src.ase_dataset import AseSqliteDataset, BufferData
from src.trainer import OrbLoss

## 数据集说明：MPtrj Dataset

### 基本信息

* 数据集名称：mptrj_ase.db
* 数据来源：Materials Project
* 目标体系：无机晶体材料（含 GGA 与 GGA+U 计算）
* 数据格式：.db

### 数据内容概览

该数据集从 Materials Project 所有 GGA/GGA+U 静态与弛豫计算轨迹中解析获得，经过去重与兼容性筛选，包含完整优化路径中的中间构型。所有能量经 MP2020 兼容性校正，确保 GGA 与 GGA+U 统一基准。适用于通用原子间势（UIP）的预训练与微调，支持能量、力、应力与磁矩联合建模。

### 加载配置

设置运行环境与上下文，根据并行模式（单机或数据并行）配置 MindSpore 的运行模式、设备和随机种子。

In [ ]:
class Args:
    def __init__(self):
        self.config = "configs/config.yaml"
        self.device_target = "Ascend"
        self.device_id = 0
        self.parallel_mode = "NONE"

args = Args()

if args.parallel_mode.upper() == "DATA_PARALLEL":
    ms.set_context(
        mode=context.PYNATIVE_MODE,
        device_target=args.device_target,
        pynative_synchronize=True,
    )
    # Set parallel context
    ms.set_auto_parallel_context(parallel_mode=ms.ParallelMode.DATA_PARALLEL, gradients_mean=True)
    init()
    ms.set_seed(1)
else:
    ms.set_context(
        mode=context.PYNATIVE_MODE,
        device_target=args.device_target,
        device_id=args.device_id,
        pynative_synchronize=True,
    )

configs = utils.load_cfg(args.config)
warnings.filterwarnings("ignore")

定义微调训练函数 `finetune`，实现单次训练循环：前向、反向、优化、日志记录、梯度裁剪、学习率调度等。

In [ ]:
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


def finetune(
        model: nn.Cell,
        loss_fn: Optional[nn.Cell],
        optimizer: nn.Optimizer,
        train_dataloader: ds.GeneratorDataset,
        val_dataloader: ds.GeneratorDataset,
        lr_scheduler: Optional[ms.experimental.optim.lr_scheduler] = None,
        clip_grad: Optional[float] = None,
        log_freq: float = 10,
        parallel_mode: str = "NONE",
):
    """Train for a fixed number of steps.

    Args:
        model: The model to optimize.
        loss_fn: The loss function to use.
        optimizer: The optimizer to use for the model.
        train_dataloader: A Dataloader, which may be infinite if num_steps is passed.
        val_dataloader: A Dataloader for validation.
        lr_scheduler: Optional, a Learning rate scheduler for modifying the learning rate.
        clip_grad: Optional, the gradient clipping threshold.
        log_freq: The logging frequency for step metrics.
        parallel_mode: The parallel mode to use, e.g., "DATA_PARALLEL" or "NONE".

    Returns
        A dictionary of metrics.
    """
    if clip_grad is not None:
        hook_handles = utils.gradient_clipping(model, clip_grad)

    train_metrics = utils.ScalarMetricTracker()
    val_metrics = utils.ScalarMetricTracker()

    epoch_metrics = {
        "data_time": 0.0,
        "train_time": 0.0,
    }

    # Get gradient function
    grad_fn = ms.value_and_grad(loss_fn.loss, None, optimizer.parameters, has_aux=True)
    if parallel_mode == "DATA_PARALLEL":
        grad_reducer = nn.DistributedGradReducer(optimizer.parameters)

    # Define function of one-step training
    def train_step(data, label=None):
        (loss, val_logs), grads = grad_fn(data, label)
        if parallel_mode == "DATA_PARALLEL":
            grads = grad_reducer(grads)
        optimizer(grads)
        return loss, val_logs

    step_begin = timeit.default_timer()
    for i, batch in enumerate(train_dataloader):
        epoch_metrics["data_time"] += timeit.default_timer() - step_begin
        # Reset metrics so that it reports raw values for each step but still do averages on
        # the gradient accumulation.
        if i % log_freq == 0:
            train_metrics.reset()

        model.set_train()
        loss, train_logs = train_step(batch)

        epoch_metrics["train_time"] += timeit.default_timer() - step_begin
        train_metrics.update(epoch_metrics)
        train_metrics.update(train_logs)

        if ops.isnan(loss):
            raise ValueError("nan loss encountered")

        if lr_scheduler is not None:
            lr_scheduler.step()
        step_begin = timeit.default_timer()

    if clip_grad is not None:
        for h in hook_handles:
            h.remove()

    # begin evaluation
    model.set_train(False)
    val_iter = iter(val_dataloader)
    val_batch = next(val_iter)
    loss, val_logs = loss_fn.loss(val_batch)
    val_metrics.update(val_logs)

    return train_metrics.get_metrics(), val_metrics.get_metrics()

构建数据加载器 `build_loader`，从 `.db` 文件加载 ASE 数据集，支持数据增强、批处理、并行切分等。

In [ ]:
def build_loader(
        dataset_path: str,
        num_workers: int,
        batch_size: int,
        augmentation: Optional[bool] = True,
        target_config: Optional[Dict] = None,
        shuffle: Optional[bool] = True,
        parallel_mode: str = "NONE",
        **kwargs,
) -> ds.GeneratorDataset:
    """Builds the dataloader from a config file.

    Args:
        dataset_path: Dataset path.
        num_workers: The number of workers for each dataset.
        batch_size: The batch_size config for each dataset.
        augmentation: If rotation augmentation is used.
        target_config: The target config.
        shuffle: If the dataset should be shuffled.
        parallel_mode: The parallel mode to use, e.g., "DATA_PARALLEL" or "NONE".

    Returns:
        The Dataloader.
    """
    log_loading = f"Loading datasets: {dataset_path} with {num_workers} workers. "
    dataset = AseSqliteDataset(
        dataset_path, target_config=target_config, augmentation=augmentation, **kwargs
    )

    log_loading += f"Total dataset size: {len(dataset)} samples"
    logging.info(log_loading)

    dataset = BufferData(dataset, shuffle=shuffle)
    if parallel_mode == "DATA_PARALLEL":
        rank_id = get_rank()
        rank_size = get_group_size()
        dataloader = [
            [dataset[j] for j in range(i, min(i + batch_size, len(dataset)))] \
                for i in range(0, len(dataset), batch_size)
        ]
        dataloader = [
            base.batch_graphs(
                data[rank_id*len(data)//rank_size : (rank_id+1)*len(data)//rank_size]
            ) for data in dataloader
        ]
    else:
        dataloader = [
            base.batch_graphs(
                [dataset[j] for j in range(i, min(i + batch_size, len(dataset)))]
            ) for i in range(0, len(dataset), batch_size)
        ]

    return dataloader

主训练流程函数 `run`，加载预训练模型、构建数据加载器、设置优化器、执行多 epoch 微调训练并保存检查点。

In [ ]:
def run(args, parallel_mode="NONE"):
    """Training Loop.

    Args:
        config (DictConfig): Config for training loop.
        parallel_mode (str): The parallel mode to use, e.g., "DATA_PARALLEL" or "NONE".
    """
    utils.seed_everything(args.random_seed)

    # Load dataset
    train_loader = build_loader(
        dataset_path=args.train_data_path,
        num_workers=args.num_workers,
        batch_size=args.batch_size,
        target_config={"graph": ["energy", "stress"], "node": ["forces"]},
        augmentation=True,
    )
    val_loader = build_loader(
        dataset_path=args.val_data_path,
        num_workers=args.num_workers,
        batch_size=1000,
        target_config={"graph": ["energy", "stress"], "node": ["forces"]},
        augmentation=False,
        shuffle=False,
    )
    num_steps = len(train_loader)

    # Instantiate model
    pretrained_weights_path = os.path.join(args.checkpoint_path, "orb-mptraj-only-v2.ckpt")
    model = pretrained.orb_mptraj_only_v2(pretrained_weights_path)
    loss_fn = OrbLoss(model)
    model_params = sum(p.size for p in model.trainable_params() if p.requires_grad)
    logging.info("Model has %d trainable parameters.", model_params)

    total_steps = args.max_epochs * num_steps
    optimizer, lr_scheduler = utils.get_optim(args.lr, total_steps, model)

    # Fine-tuning loop
    start_epoch = 0
    train_time = timeit.default_timer()
    for epoch in range(start_epoch, args.max_epochs):
        train_metrics, val_metrics = finetune(
            model=model,
            loss_fn=loss_fn,
            optimizer=optimizer,
            train_dataloader=train_loader,
            val_dataloader=val_loader,
            lr_scheduler=lr_scheduler,
            clip_grad=args.gradient_clip_val,
            parallel_mode=parallel_mode,
        )
        print(f'Epoch: {epoch}/{args.max_epochs}, \n train_metrics: {train_metrics}\n val_metrics: {val_metrics}')

        # Save checkpoint from last epoch
        if epoch == args.max_epochs - 1:
            # create ckpts folder if it does not exist
            if not os.path.exists(args.checkpoint_path):
                os.makedirs(args.checkpoint_path)
            if parallel_mode == "DATA_PARALLEL":
                rank_id = get_rank()
                rank_size = get_group_size()
                ms.save_checkpoint(
                    model,
                    os.path.join(
                        args.checkpoint_path,
                        f"orb-ft-parallel[{rank_id}-{rank_size}]-checkpoint_epoch{epoch}.ckpt"
                    ),
                )
            else:
                ms.save_checkpoint(
                    model,
                    os.path.join(args.checkpoint_path, f"orb-ft-checkpoint_epoch{epoch}.ckpt"),
                )
            logging.info("Checkpoint saved to %s", args.checkpoint_path)
    logging.info("Training time: %.5f seconds", timeit.default_timer() - train_time)

In [ ]:
run(configs, args.parallel_mode)